In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 03 E2: DenseNet-121 with the tier head (Stage 5c)\n",
    "\n",
    "Same recipe as E1 (seed 42) plus two extra outputs trained on `tier >= URGENT` and `tier == IMMEDIATE`. Runs on Kaggle with a **GPU** and **Internet on**; add the `Chestradix-nih224` dataset as input and attach the `GITHUB_TOKEN` secret."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Setup"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os, subprocess, sys, glob\n",
    "from kaggle_secrets import UserSecretsClient\n",
    "\n",
    "token = UserSecretsClient().get_secret(\"GITHUB_TOKEN\")\n",
    "REPO_URL = f\"https://{token}@github.com/cmihiranga099/ChestRadix.git\"\n",
    "BRANCH = \"stage-5c-tier-head\"\n",
    "if not os.path.exists(\"/kaggle/working/chestradix\"):\n",
    "    subprocess.run([\"git\", \"clone\", \"--branch\", BRANCH, REPO_URL, \"/kaggle/working/chestradix\"], check=True)\n",
    "os.chdir(\"/kaggle/working/chestradix\")\n",
    "subprocess.run([sys.executable, \"-m\", \"pip\", \"install\", \"-q\", \"scikit-learn\", \"pandas\", \"scipy\"], check=True)\n",
    "\n",
    "hits = glob.glob(\"/kaggle/input/**/images_224.npy\", recursive=True)\n",
    "assert hits, \"Dataset not attached: add Chestradix-nih224 under Input\"\n",
    "os.environ[\"CHESTRADIX_DATA_DIR\"] = os.path.dirname(hits[0])\n",
    "\n",
    "import torch\n",
    "print(\"torch\", torch.__version__, \"| CUDA:\", torch.cuda.is_available(),\n",
    "      torch.cuda.get_device_name(0) if torch.cuda.is_available() else \"\")\n",
    "from src.config import locate_processed_dir\n",
    "print(\"data:\", locate_processed_dir())"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Smoke test with the tier head (about 2 minutes)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "!python -m src.train --tier-head --run-name smoke_tier --epochs 1 --max-batches 20 --num-workers 2"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. E2 headline run (about 1.5 to 2 hours)\n",
    "\n",
    "Same seed and settings as E1 so the only difference is the tier head. `--time-budget-min 420` stops cleanly after the epoch that passes 7 hours."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "!python -m src.train --tier-head --backbone densenet121 --seed 42 --run-name e2_densenet121_tier_s42 --epochs 15 --time-budget-min 420 --num-workers 2"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Calibrate, evaluate and triage (validation fits, test reports)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "RUN = \"/kaggle/working/runs/e2_densenet121_tier_s42\"\n",
    "!python -m src.calibration --run-dir {RUN}\n",
    "!python -m src.evaluate --run-dir {RUN} --logits-name logits_calibrated --n-boot 1000\n",
    "!python -m src.triage --run-dir {RUN} --logits-name logits_calibrated --recall-floor 0.80 --n-boot 1000"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import json\n",
    "print(open(f\"{RUN}/summary.json\").read())"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Package results for download\n",
    "\n",
    "`best.pt` is excluded (download it separately from the Output tab if needed)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "!cd /kaggle/working && zip -r -q runs_e2.zip runs/e2_densenet121_tier_s42 -x \"*/best.pt\" && ls -lh runs_e2.zip"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}